In [ ]:
%run ./_common

In [ ]:
SA = SWSAcademyHelper()  # Create the SA object
SA.init()                                           # Performs basic intialization including creating schemas and catalogs

In [ ]:
# Import necessary libraries
import os
import configparser
import ipywidgets as widgets
from urllib.parse import urlparse, urlunsplit

# Token Class Definition
class Token:
    def __init__(self, config_path='./M02 - Architecting MLOps Solutions/var/credentials.cfg'):
        self.config_path = config_path
        self.token = self.get_credentials()

    def get_credentials(self):
        # Check if the file exists
        if not os.path.exists(self.config_path):
            # No saved PAT (e.g. "0 - Generate Tokens" was skipped, or running as a job):
            # use the token of the current notebook session instead.
            print(f"No credentials file at {self.config_path}; using the notebook session token.")
            return dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

        config = configparser.ConfigParser()
        config.read(self.config_path)

        if 'DEFAULT' not in config:
            raise KeyError("The section 'DEFAULT' was not found in the configuration file.")

        if 'db_token' not in config['DEFAULT']:
            raise KeyError("The key 'db_token' was not found in the 'DEFAULT' section of the configuration file.")

        token = config['DEFAULT']['db_token']
        
        # Never print the full token; show only enough to confirm which one was loaded
        print(f"db_token loaded: {token[:4]}...{token[-4:]}")

        return token

In [ ]:
def init_DA(name, reset=True, pipeline=False):

    if pipeline:
        SA.paths.stream_source = f"{SA.paths.working_dir}/stream_source"
        SA.paths.storage_location = f"{SA.paths.working_dir}/storage_location"
        SA.pipeline_name = f"{SA.unique_name(sep='-')}: ETL Pipeline - {name.upper()}"
        SA.lookup_db = f"{SA.unique_name(sep='_')}_lookup"

        if reset:  # create lookup tables
            import pandas as pd

            spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SA.lookup_db}")
            SA.clone_source_table(f"{SA.lookup_db}.date_lookup", source_name="date-lookup")
            SA.clone_source_table(f"{SA.lookup_db}.user_lookup", source_name="user-lookup")

            df = spark.createDataFrame(pd.DataFrame([
                ("device_id_not_null", "device_id IS NOT NULL", "validity", "bpm"),
                ("device_id_valid_range", "device_id > 110000", "validity", "bpm"),
                ("heartrate_not_null", "heartrate IS NOT NULL", "validity", "bpm"),
                ("user_id_not_null", "user_id IS NOT NULL", "validity", "workout"),
                ("workout_id_not_null", "workout_id IS NOT NULL", "validity", "workout"),
                ("action_not_null","action IS NOT NULL","validity","workout"),
                ("user_id_not_null","user_id IS NOT NULL","validity","user_info"),
                ("update_type_not_null","update_type IS NOT NULL","validity","user_info")
            ], columns=["name", "condition", "tag", "topic"]))
            df.write.mode("overwrite").saveAsTable(f"{SA.lookup_db}.rules")
        
    return SA

In [ ]:
# Initialize Lesson with Configurations
LESSON = "generate_tokens"
SA = init_DA(LESSON, reset=False)

In [ ]:
# Create Credentials File Method
@SWSAcademyHelper.add_method
def load_credentials(self):
    import configparser
    c = configparser.ConfigParser()
    c.read(filenames=f"var/credentials.cfg")
    try:
        token = c.get('DEFAULT', 'db_token')
        host = c.get('DEFAULT', 'db_instance')
        os.environ["DATABRICKS_HOST"] = host
        os.environ["DATABRICKS_TOKEN"] = token
    except:
        token = ''
        host = f'https://{spark.conf.get("spark.databricks.workspaceUrl")}'

    return token, host    
@SWSAcademyHelper.add_method
def create_credentials_file(self, db_token, db_instance):
    contents = f"""
db_token = "{db_token}"
db_instance = "{db_instance}"
    """
    self.write_to_file(contents, "credentials.py")
None

# Get Credentials Method
@SWSAcademyHelper.add_method
def get_credentials(self):
    (current_token, current_host) = self.load_credentials()

    @widgets.interact(host=widgets.Text(description='Host:',
                                        placeholder='Paste workspace URL here',
                                        value=current_host,
                                        continuous_update=False),
                      token=widgets.Password(description='Token:',
                                             placeholder='Paste PAT here',
                                             value=current_token,
                                             continuous_update=False)
    )
    def _f(host='', token=''):
        u = urlparse(host)
        host = urlunsplit((u.scheme, u.netloc, '', '', ''))

        if host and token:
            os.environ["DATABRICKS_HOST"] = host
            os.environ["DATABRICKS_TOKEN"] = token

            contents = f"""
[DEFAULT]
db_token = {token}
db_instance = {host}
            """
            # Save credentials in the new location
            os.makedirs('./var', exist_ok=True)
            with open("./var/credentials.cfg", "w") as f:
                print(f"Credentials stored ({f.write(contents)} bytes written).")

None

In [ ]:
# Initialize the next lesson and load credentials
LESSON = "using_cli"
SA = init_DA(LESSON, reset=False, pipeline=True)

# Load Credentials
(db_token, db_instance) = SA.load_credentials()